In [3]:
import glob
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ---------- Load data ----------
matches = glob.glob('/kaggle/input/datasets/sachinarkesh/co2-emission/co2_emissions.csv', recursive=True)
USING_DEMO = not matches

if matches:
    print("Using:", matches[0])
    df = pd.read_csv(matches[0])
else:
    print("WARNING: co2_emissions.csv not found. Using FAKE demo data. "
          "Do not submit this chart.")
    rng = np.random.default_rng(0)
    base = {'China': 3400, 'India': 1000, 'Japan': 1200, 'South Korea': 450, 'Indonesia': 300}
    growth = {'China': 0.06, 'India': 0.045, 'Japan': -0.01, 'South Korea': 0.01, 'Indonesia': 0.04}
    rows = [(c, 'Asia', y, base[c] * (1 + growth[c]) ** (y - 2000) * rng.normal(1, 0.01), 1.0)
            for c in base for y in range(2000, 2023)]
    df = pd.DataFrame(rows, columns=['Country', 'Region', 'Year', 'CO2_Mt', 'CO2_per_capita'])

# ---------- Settings ----------
HIGHLIGHT = 'China'
HIGHLIGHT_COLOR = '#D62728'
GREY = '#EEEEEE'

# ---------- Prepare data ----------
asia = df[df['Region'] == 'Asia'].sort_values(['Country', 'Year'])
hl = asia[asia['Country'] == HIGHLIGHT]
start, end = hl.iloc[0], hl.iloc[-1]
multiple = end['CO2_Mt'] / start['CO2_Mt']

# ---------- Build chart ----------
fig = go.Figure()

for country, g in asia[asia['Country'] != HIGHLIGHT].groupby('Country'):
    fig.add_trace(go.Scatter(x=g['Year'], y=g['CO2_Mt'], mode='lines',
                             line=dict(color=GREY, width=1.5),
                             hoverinfo='skip', showlegend=False))

fig.add_trace(go.Scatter(x=hl['Year'], y=hl['CO2_Mt'], mode='lines',
                         line=dict(color=HIGHLIGHT_COLOR, width=4),
                         showlegend=False,
                         hovertemplate='%{x}: %{y:,.0f} Mt<extra></extra>'))

fig.add_annotation(x=end['Year'], y=end['CO2_Mt'],
                   text=f"<b>{HIGHLIGHT}</b><br>{end['CO2_Mt']:,.0f} Mt",
                   showarrow=False, xanchor='left', xshift=8,
                   font=dict(color=HIGHLIGHT_COLOR, size=14))

title = (f"<b>{HIGHLIGHT}'s CO2 emissions grew {multiple:.1f}x since "
         f"{int(start['Year'])}, pulling far ahead of the rest of Asia</b>")
if USING_DEMO:
    title = "<b>[DEMO DATA] </b>" + title

fig.update_layout(
    title=dict(text=title, x=0, xanchor='left', font=dict(size=20)),
    template='simple_white', plot_bgcolor='white', paper_bgcolor='white',
    margin=dict(l=60, r=130, t=90, b=60), width=900, height=520,
    xaxis=dict(title=None, showgrid=False, showline=False, tickangle=0, dtick=2, ticks=''),
    yaxis=dict(title='CO2 emissions (Mt)', showgrid=True, gridcolor='#F2F2F2',
               gridwidth=0.5, showline=False, zeroline=False, ticks='', tickformat=','),
    showlegend=False
)

fig.show()

Using: /kaggle/input/datasets/sachinarkesh/co2-emission/co2_emissions.csv
